# Cropy 10클래스 전체 데이터 448px 재학습 (서버 교체 후보)

작업자 분리 시험셋 실험에서 입력 448px가 무노균병 recall을 74%→88%로 올렸습니다(macro F1 0.899→0.917, 나빠진 클래스 없음).
이 노트북은 **서버 모델과 같은 전체 데이터(공식 train 37,753장 / val 4,635장, 10클래스)** 로 입력만 448px인 ResNet50을 학습해 교체 후보를 만듭니다.

- 데이터: `cropy_full_512.zip` + `aihub_other_v1.zip` + `coco_other_v1.zip` (서버 모델 학습 때와 같은 구성, 재분할 없음).
- 학습: ResNet50(ImageNet 가중치), 448px, 흐림 증강, 클래스 균형, lr 1e-4, **8 epoch** (시험셋 실험에서 4 epoch 이후 평평했음). T4 기준 epoch당 약 35분, 총 4.5~5시간.
- 결과: `MyDrive/cropy_testset/full448_v1/` (`best_model.pth`, `val_report.json`).
- 마지막 셀이 공식 val 4,635장으로 새 모델과 현재 서버 모델(256px)을 비교합니다. 이 val은 학습 작업자와 겹치므로 **회귀 확인용**이며, 일반화 근거는 시험셋 실험(`hires448_v1`)입니다.

**GPU 런타임** 에서 위에서부터 차례로 실행하세요. 끊기면 다시 위에서부터 실행하면 마지막 epoch부터 이어집니다.

In [ ]:
from pathlib import Path
import csv, hashlib, json, re, shutil, subprocess, sys, zipfile
import numpy as np, torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')
TS = MY / 'cropy_testset'
RUN = TS / 'full448_v1'
WORK = Path('/content/cropy_full448')
DATA = WORK / 'dataset'
EPOCHS, BATCH_SIZE, LR, IMG_SIZE = 8, 16, 1e-4, 448
EXPECT_IMAGES = (37753, 4635)
EXPECTED_BASE_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = 'efe7034c6acd0544416e0cdb94c8fe748081921d73cf2fd1513fbc9b55e292a4'
SOURCES = {'base': MY / 'cropy_full_512/cropy_full_512.zip', 'code': MY / 'cropy_other10/training_code_other10.zip',
           'aihub': MY / 'cropy_other10/aihub_other_v1.zip', 'coco': MY / 'cropy_other10/coco_other_v1.zip',
           'server': MY / 'cropy_other10/cropy_other10_handoff_20261027/full512_resnet50_other10_v1.pth',
           'trainer': TS / 'train_full_colab_convnext.py'}
for path in SOURCES.values():
    assert path.is_file(), f'Drive에 파일이 없습니다: {path}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비 (공식 분할 그대로)\nDrive에서 로컬로 복사해 풀고, 10클래스 사진 수가 서버 모델 학습 때와 같은지 확인합니다.

In [ ]:
EXPECT = {"train": {"0": 967, "1": 912, "2": 462, "3": 227, "4": 801, "5": 456, "6": 8575, "7": 10797, "8": 9316, "9": 5240},
          "val": {"0": 117, "1": 112, "2": 76, "3": 24, "4": 110, "5": 37, "6": 1110, "7": 1301, "8": 1183, "9": 565}}

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(src, dest, expected=None):
    local = WORK / src.name
    if not local.exists():
        shutil.copyfile(src, local)
    if expected:
        assert digest(local) == expected, f'해시 불일치: {src.name}'
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (dest / item.filename).resolve().is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack(SOURCES['code'], WORK / 'code', EXPECTED_CODE_SHA256)
if not (WORK / 'data_done').exists():
    shutil.rmtree(DATA, ignore_errors=True)
    unpack(SOURCES['base'], WORK, EXPECTED_BASE_SHA256)
    unpack(SOURCES['aihub'], WORK)
    unpack(SOURCES['coco'], WORK)
    (WORK / 'data_done').touch()

counts = {s: {c.name: len(list(c.iterdir())) for c in sorted((DATA / 'images' / s).iterdir())} for s in ('train', 'val')}
for s, cs in counts.items():
    print(s, sum(cs.values()), cs)
for s, cs in EXPECT.items():
    for c, n in cs.items():
        got = counts[s][[k for k in counts[s] if k.startswith(c + '_')][0]]
        assert got == n, f'{s}/{c}: {got} != 기대 {n}'
assert tuple(sum(counts[s].values()) for s in ('train', 'val')) == EXPECT_IMAGES
from torchvision.datasets import ImageFolder
train_ds, val_ds = ImageFolder(DATA / 'images/train'), ImageFolder(DATA / 'images/val')
assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 10, train_ds.classes
(RUN / 'data_counts.json').write_text(json.dumps(counts, ensure_ascii=False, indent=1), encoding='utf-8')
print('10클래스 사진 수가 서버 모델 학습 구성과 일치합니다:', EXPECT_IMAGES)

## 2. ResNet50 448px 학습 (ImageNet에서 시작)\n학습 스크립트는 서버 모델 코드에 `--arch` 선택만 추가된 판입니다. 메모리 부족(CUDA out of memory)이 나면 1번 셀의 BATCH_SIZE를 8로 줄이고 다시 실행하세요.

In [ ]:
shutil.copyfile(SOURCES['trainer'], WORK / 'code/train_full_colab.py')
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(DATA / 'images'), '--output-dir', str(RUN / 'main'),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
    '--num-workers', '2', '--img-size', str(IMG_SIZE), '--arch', 'resnet50', '--blur-aug',
    '--expect-images', str(EXPECT_IMAGES[0]), str(EXPECT_IMAGES[1])]
if (RUN / 'main/last_checkpoint.pth').exists():
    command.append('--resume')
print('이어 학습:', '--resume' in command)
subprocess.run(command, check=True)
shutil.copyfile(RUN / 'main/best_model.pth', RUN / 'best_model.pth')
print('SHA256:', digest(RUN / 'best_model.pth'))

## 3. 공식 val로 회귀 확인\n새 448px 모델과 현재 서버 모델(256px)을 각자 입력 크기로 val 4,635장에 평가합니다. 작물 사진이 9_기타로 가는 수와 질병별 recall을 봅니다.

In [ ]:
from PIL import Image
from torchvision import transforms

def load(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    sys.path.insert(0, str(WORK / 'code'))
    from train_full_colab import build_model
    model = build_model(ckpt.get('arch', 'resnet50'), ckpt['num_classes'], pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.cuda().eval(), ckpt

def predict(model, size, files):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    out = []
    for i in range(0, len(files), 64):
        batch = files[i:i + 64]
        with torch.no_grad():
            out.append(model(torch.stack([tf(Image.open(p).convert('RGB')) for p in batch]).cuda()).softmax(1).cpu().numpy())
    return np.concatenate(out)

def metrics(y, pred, ids):
    cm = np.zeros((len(ids), len(ids)), dtype=np.int64)
    for a, b in zip(y, pred):
        cm[ids.index(a), ids.index(b)] += 1
    recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
    precision = cm.diagonal() / np.maximum(cm.sum(0), 1)
    f1 = 2 * recall * precision / np.maximum(recall + precision, 1e-9)
    crop_to_other = int(cm[:9, 9].sum())
    return dict(n=int(len(y)), accuracy=round(float((pred == y).mean()), 4), macro_f1=round(float(f1.mean()), 4),
                recall={str(c): round(float(r), 4) for c, r in zip(ids, recall)}, crop_to_other=crop_to_other, confusion=cm.tolist())

files = sorted((DATA / 'images/val').glob('*/*'))
y = np.array([int(p.parent.name.split('_')[0]) for p in files])
report = {}
for name, path in (('full448', RUN / 'best_model.pth'), ('server256', SOURCES['server'])):
    model, ckpt = load(path)
    ids = [int(c.split('_', 1)[0]) for c in ckpt['classes']]
    probs = predict(model, ckpt.get('img_size', 256), files)
    pred = np.array(ids)[probs.argmax(1)]
    report[name] = metrics(y, pred, ids)
    report[name]['epoch'] = int(ckpt['epoch'])
    report[name]['img_size'] = int(ckpt.get('img_size', 256))
    with (RUN / f'predictions_{name}_val.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['file', 'label'] + [f'p{i}' for i in ids])
        for p, pr in zip(files, probs):
            w.writerow([p.name, int(p.parent.name.split('_')[0])] + [round(float(v), 5) for v in pr])
    r = report[name]
    print(f"{name}: 정확도 {r['accuracy']:.2%}, macro F1 {r['macro_f1']:.3f}, 무검은무늬 {r['recall']['2']:.1%}, 무노균 {r['recall']['3']:.1%}, 작물→기타 {r['crop_to_other']}장")
    del model
report['best_model_sha256'] = digest(RUN / 'best_model.pth')
(RUN / 'val_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
print('저장:', RUN / 'val_report.json')